# 02 - Fine-tuning Qwen2-VL-2B with DoRA

**Goal:** Adapt `Qwen/Qwen2-VL-2B-Instruct` to Lithuanian cultural VQA using our reviewed dataset.

**Stack:**
- Pure PyTorch + PyTorch Lightning (`LightningModule` + `Trainer`)
- LoRA rank 16 with `use_dora=True` (DoRA = magnitude+direction decomposition, the advanced technique)
- Train: LLM attention projections (q/k/v/o) + MLP visual projector
- Freeze: ViT vision encoder + LLM MLP layers
- bf16 mixed precision, gradient checkpointing, batch=1 x grad_accum=8 -> effective batch 8
- 8-bit `paged_adamw` optimizer (memory-efficient, recommended choice with QLoRA)

## 1. Setup

In [3]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [2]:
# Pinned versions known to work together for Qwen2-VL + QLoRA + DoRA + Lightning (Apr 2026 ecosystem)
!pip -q install "transformers>=4.45,<4.50" "accelerate>=0.34" "peft>=0.13" \
    "bitsandbytes>=0.44" "qwen-vl-utils>=0.0.8" "pillow" "sentencepiece" \
    "pytorch-lightning>=2.3,<2.5" "torchmetrics>=1.4" "torchao>=0.16.0"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 151.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 41.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 815.2/815.2 kB 61.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 70.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 125.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 46.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 109.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.3/36.3 MB 65.1 MB/s eta 0:00:00


- **Transformers 4.45+**: Required for native `Qwen2VL` support.
- **PEFT 0.13+**: Supports the `use_dora=True` flag for LoRA.
- **Bitsandbytes**: Essential for 8-bit optimization to keep the 2B model + gradients within the 24GB VRAM limit of an L4 GPU.
- **Torchao**: Used for low-precision kernels and optimization utilities.

In [1]:
import os, json, math, random
from pathlib import Path
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from collections import Counter
import pytorch_lightning as pl
from pytorch_lightning.callbacks import ModelCheckpoint, LearningRateMonitor
from transformers import (
    Qwen2VLForConditionalGeneration,
    AutoProcessor
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
import bitsandbytes as bnb

PROJECT_DIR = Path("/content/drive/MyDrive/VU_DL_task2")
DATASET_DIR = PROJECT_DIR / "dataset"
CKPT_DIR = PROJECT_DIR / "checkpoints" / "qlora_dora"
CKPT_DIR.mkdir(parents=True, exist_ok=True)

MODEL_ID = "Qwen/Qwen2-VL-2B-Instruct"
SEED = 42
pl.seed_everything(SEED, workers=True)
torch.set_float32_matmul_precision('high')
print(
    "CUDA:",
    torch.cuda.is_available(),
    torch.cuda.get_device_name(0) if torch.cuda.is_available() else "",
)

INFO:lightning_fabric.utilities.seed:Seed set to 42


CUDA: True NVIDIA L4


## 2. Load dataset

In [2]:
rows = [
    json.loads(l)
    for l in open(DATASET_DIR / "dataset.jsonl", encoding="utf-8")
    if l.strip()
]
print("total QA:", len(rows))
print("per split:", Counter(r["split"] for r in rows))
print("per qa_type:", Counter(r["qa_type"] for r in rows))

train_rows = [r for r in rows if r["split"] == "train"]
val_rows = [r for r in rows if r["split"] == "val"]
print("train:", len(train_rows), " val:", len(val_rows))

total QA: 888
per split: Counter({'train': 728, 'val': 80, 'test': 80})
per qa_type: Counter({'identification': 222, 'visual': 222, 'cultural': 222, 'adversarial': 222})
train: 728  val: 80


## 3. Load processor + model and attach DoRA adapters

In [3]:
processor = AutoProcessor.from_pretrained(MODEL_ID, use_fast=False)
# Cap visual tokens so each sample stays well under L4 VRAM. 256-768 patches.
processor.image_processor.min_pixels = 256 * 28 * 28
processor.image_processor.max_pixels = 768 * 28 * 28

print("Loading base model in pure bfloat16...")
base = Qwen2VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map={"": 0},
)
base.config.use_cache = False

# Natively enable gradient checkpointing to save VRAM (replaces prepare_model_for_kbit_training)
base.gradient_checkpointing_enable()

lora_cfg = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    use_dora=True,  # We use DoRA technique
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
)
peft_model = get_peft_model(base, lora_cfg)

# Also unfreeze the small visual->LLM projector ('merger') so vision can adapt to LT context.
for n, p in peft_model.named_parameters():
    if "visual.merger" in n:
        # Cast to bf16
        p.data = p.data.to(torch.bfloat16)
        p.requires_grad = True

peft_model.print_trainable_parameters()

Loading base model in pure bfloat16...


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

trainable params: 38,546,432 || all params: 2,213,444,096 || trainable%: 1.7415


### Architectural Choices for Fine-tuning
1. **DoRA (Weight-Decomposition Low-Rank Adaptation)**: Unlike standard LoRA which only adapts the direction of weights, DoRA decomposes weights into magnitude and direction. This has been shown to more closely mimic full fine-tuning performance while remaining parameter-efficient.
2. **Targeting Attention Layers**: We focus on `q, k, v, o` projections in the LLM. These layers hold the most "knowledge" and linguistic capability.
3. **Unfreezing the 'Merger'**: The `visual.merger` is the bridge between the ViT encoder and the LLM. By unfreezing it, we allow the model to learn how to better map Lithuanian-specific visual features into the language space.

## 4. Dataset + collator

Qwen2-VL uses a chat template with explicit `<|image_pad|>` slots. Labels mask everything except the assistant answer so the loss is computed only on the model's response.

In [4]:
SYSTEM_LT = (
    "Tu esi lietuvių kultūros žinovas. Atsakyk trumpai ir tiksliai lietuvių kalba "
    "remdamasis pateiktu vaizdu."
)
IGNORE_INDEX = -100


def build_messages(question):
    return [
        {"role": "system", "content": [{"type": "text", "text": SYSTEM_LT}]},
        {
            "role": "user",
            "content": [
                {"type": "image"},
                {"type": "text", "text": question},
            ],
        },
    ]


class LTCulturalVQA(Dataset):
    def __init__(self, rows):
        self.rows = rows

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, i):
        r = self.rows[i]
        return {
            "image": Image.open(r["image_path"]).convert("RGB"),
            "question": r["question"],
            "answer": r["answer"],
        }


class QwenVLCollator:
    def __init__(self, processor):
        self.processor = processor
        self.pad_id = processor.tokenizer.pad_token_id

    def __call__(self, batch):
        input_ids_list, labels_list, pixel_list, grid_list = [], [], [], []
        for ex in batch:
            msgs = build_messages(ex["question"])
            prompt_text = self.processor.apply_chat_template(
                msgs, add_generation_prompt=True, tokenize=False
            )
            full_text = prompt_text + ex["answer"] + self.processor.tokenizer.eos_token

            prompt_inputs = self.processor(
                text=[prompt_text], images=[ex["image"]], return_tensors="pt"
            )
            full_inputs = self.processor(
                text=[full_text], images=[ex["image"]], return_tensors="pt"
            )

            input_ids = full_inputs["input_ids"][0]
            prompt_len = prompt_inputs["input_ids"].shape[1]
            labels = input_ids.clone()
            labels[:prompt_len] = IGNORE_INDEX

            input_ids_list.append(input_ids)
            labels_list.append(labels)
            pixel_list.append(full_inputs["pixel_values"])
            grid_list.append(full_inputs["image_grid_thw"])

        maxlen = max(x.size(0) for x in input_ids_list)

        def pad(t, value):
            return torch.cat(
                [t, torch.full((maxlen - t.size(0),), value, dtype=t.dtype)]
            )

        input_ids = torch.stack([pad(x, self.pad_id) for x in input_ids_list])
        labels = torch.stack([pad(x, IGNORE_INDEX) for x in labels_list])
        attention_mask = (input_ids != self.pad_id).long()

        return {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "labels": labels,
            "pixel_values": torch.cat(pixel_list, dim=0),
            "image_grid_thw": torch.cat(grid_list, dim=0),
        }


train_ds = LTCulturalVQA(train_rows)
val_ds = LTCulturalVQA(val_rows)
collate = QwenVLCollator(processor)

# Smoke test on one batch
sample = collate([train_ds[0]])
print({k: (v.shape if hasattr(v, "shape") else v) for k, v in sample.items()})

{'input_ids': torch.Size([1, 852]), 'attention_mask': torch.Size([1, 852]), 'labels': torch.Size([1, 852]), 'pixel_values': torch.Size([2976, 1176]), 'image_grid_thw': torch.Size([1, 3])}


## 5. LightningModule

Wraps the PEFT model. `training_step` / `validation_step` just forward the batch (HF model returns loss when `labels` are provided). Optimizer is 8-bit `PagedAdamW8bit` from bitsandbytes — much lower memory than `torch.optim.AdamW` and recommended for LoRA.
- **PagedAdamW8bit**: We use the 'paged' version of the 8-bit optimizer. Paging allows the optimizer state to be offloaded to CPU memory if GPU memory spikes, preventing Out-Of-Memory (OOM) crashes during the backward pass.
- **Cosine Learning Rate Schedule**: Starts with a linear warmup to avoid destabilizing the pre-trained weights, followed by a cosine decay to converge smoothly.

In [5]:
class QwenVLLitModule(pl.LightningModule):
    def __init__(
        self, model, lr=2e-4, weight_decay=0.0, warmup_ratio=0.03, total_steps=1000
    ):
        super().__init__()
        self.model = model
        self.lr = lr
        self.weight_decay = weight_decay
        self.warmup_ratio = warmup_ratio
        self.total_steps = total_steps
        self.save_hyperparameters(ignore=["model"])

    def forward(self, **batch):
        return self.model(**batch)

    def training_step(self, batch, batch_idx):
        out = self.model(**batch)
        self.log("train/loss", out.loss, prog_bar=True, on_step=True, on_epoch=True)
        return out.loss

    def validation_step(self, batch, batch_idx):
        out = self.model(**batch)
        self.log(
            "val/loss",
            out.loss,
            prog_bar=True,
            on_step=False,
            on_epoch=True,
            sync_dist=True,
        )
        return out.loss

    def configure_optimizers(self):
        trainable = [p for p in self.model.parameters() if p.requires_grad]
        opt = bnb.optim.PagedAdamW8bit(
            trainable, lr=self.lr, weight_decay=self.weight_decay, betas=(0.9, 0.999)
        )
        warmup = max(1, int(self.warmup_ratio * self.total_steps))

        def lr_lambda(step):
            if step < warmup:
                return step / warmup
            progress = (step - warmup) / max(1, self.total_steps - warmup)
            return 0.5 * (1.0 + math.cos(math.pi * progress))

        sched = torch.optim.lr_scheduler.LambdaLR(opt, lr_lambda)
        return {
            "optimizer": opt,
            "lr_scheduler": {"scheduler": sched, "interval": "step"},
        }

## 6. Train

In [6]:
BATCH_SIZE = 1
GRAD_ACCUM = 8
EPOCHS = 5

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    collate_fn=collate,
    pin_memory=True,
)
val_loader = DataLoader(
    val_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    collate_fn=collate,
    pin_memory=True,
)

steps_per_epoch = math.ceil(len(train_loader) / GRAD_ACCUM)
total_steps = steps_per_epoch * EPOCHS
print(f"steps/epoch={steps_per_epoch}  total_steps={total_steps}")

lit = QwenVLLitModule(
    peft_model,
    lr=2e-4,
    weight_decay=0.0,
    warmup_ratio=0.03,
    total_steps=total_steps,
)

steps/epoch=91  total_steps=455


In [7]:
ckpt_cb = ModelCheckpoint(
    dirpath=str(CKPT_DIR / "lightning_ckpts"),
    filename="epoch{epoch:02d}-val{val/loss:.3f}",
    monitor="val/loss",
    mode="min",
    save_top_k=2,
    save_last=True,
    auto_insert_metric_name=False,
)
lr_cb = LearningRateMonitor(logging_interval="step")

trainer = pl.Trainer(
    max_epochs=EPOCHS,
    accelerator="gpu",
    devices=1,
    precision="bf16-mixed",
    accumulate_grad_batches=GRAD_ACCUM,
    gradient_clip_val=1.0,
    log_every_n_steps=10,
    callbacks=[ckpt_cb, lr_cb],
    enable_progress_bar=True,
    default_root_dir=str(CKPT_DIR),
)

INFO:pytorch_lightning.utilities.rank_zero:Using bfloat16 Automatic Mixed Precision (AMP)
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:HPU available: False, using: 0 HPUs


- **Effective Batch Size**: Since the L4 GPU can only fit `batch_size=1` for a vision-language model of this size, we use `accumulate_grad_batches=8` to simulate a batch size of 8. This provides more stable gradients for the optimizer.
- **BF16-Mixed**: Bfloat16 is preferred over FP16 for large models as it has the same dynamic range as FP32, preventing overflow issues during training.

In [8]:
trainer.fit(lit, train_loader, val_loader)

INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:pytorch_lightning.callbacks.model_summary:
  | Name  | Type                 | Params | Mode 
-------------------------------------------------------
0 | model | PeftModelForCausalLM | 2.2 B  | train
-------------------------------------------------------
38.5 M    Trainable params
2.2 B     Non-trainable params
2.2 B     Total params
8,853.776 Total estimated model params size (MB)
1234      Modules in train mode
730       Modules in eval mode


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/data.py:78: Trying to infer the `batch_size` from an ambiguous collection. The batch size we found is 1. To avoid any miscalculations, use `self.log(..., batch_size=batch_size)`.


Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=5` reached.


## 7. Save adapter to Drive

We save the PEFT (LoRA+DoRA) adapter — small (~50 MB), reloadable on top of the base model in the eval notebook.

In [9]:
# Grab the path to the best checkpoint your callback saved
best_ckpt_path = ckpt_cb.best_model_path
print(f"The model with lowest val loss was saved at: {best_ckpt_path}")

# Load those optimal weights back into your active model
checkpoint = torch.load(best_ckpt_path)
lit.load_state_dict(checkpoint['state_dict'], strict=False)
print("Best weights successfully loaded into memory!")

The model with lowest val loss was saved at: /content/drive/MyDrive/VU_DL_task2/checkpoints/qlora_dora/lightning_ckpts/epoch02-val1.422.ckpt
Best weights successfully loaded into memory!


In [10]:
FINAL_DIR = CKPT_DIR / "final"
lit.model.save_pretrained(str(FINAL_DIR))
processor.save_pretrained(str(FINAL_DIR))
print("Saved adapter to:", FINAL_DIR)

Saved adapter to: /content/drive/MyDrive/VU_DL_task2/checkpoints/qlora_dora/final


## 8. Quick sanity generation

In [12]:
# 1. Ensure model is on GPU and set to eval mode
lit.model.to("cuda")
lit.model.eval()
lit.model.config.use_cache = True

# 2. Get a sample and prepare inputs
ex = val_ds[0]
msgs = build_messages(ex["question"])
prompt = processor.apply_chat_template(msgs, add_generation_prompt=True, tokenize=False)

# 3. Process inputs and move to GPU with correct dtype
inputs = processor(text=[prompt], images=[ex["image"]], return_tensors="pt")

# Qwen2-VL requires pixel_values in bfloat16 for the visual encoder when using QLoRA
inputs = {k: v.to("cuda") if isinstance(v, torch.Tensor) else v for k, v in inputs.items()}
if "pixel_values" in inputs:
    inputs["pixel_values"] = inputs["pixel_values"].to(torch.bfloat16)

# 4. Generate with a standard torch.no_grad block
# We avoid torch.compile or dynamo blocks here to prevent the bitsandbytes CPU fallback
with torch.no_grad():
    out = lit.model.generate(
        **inputs,
        max_new_tokens=128,
        do_sample=False,
        use_cache=True
    )

gen = processor.batch_decode(
    out[:, inputs["input_ids"].shape[1] :], skip_special_tokens=True
)[0]

print("Q:", ex["question"])
print("GT:", ex["answer"])
print("PRED:", gen)

Q: Kas pavaizduota šiame vaizde?
GT: Pavaizduotas tradicinis lietuviškas bulvinių kukulių patiekalas – cepelinai (arba didžkukuliai) su mėsos įdaru.
PRED: Pavaizduoti tradiciniai lietuviški bulviniai kukuliai (cepelinai) su mėsos įdaru, patiekiamos su smulkintais grietinėmis ir patiekiamos su šviesiu kruštu.
